# Notebook principal

## Importação de bibliotecas

In [31]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from sklearn.preprocessing import MinMaxScaler

## Abertura do dataset

In [10]:
df = pd.read_csv('../data/wage.csv')
display(df.head())

,Unnamed: 0,year,age,maritl,race,education,jobclass,health,health_ins,wage
0,0,2006,18,1. Never Married,1. White,1. < HS Grad,1. Industrial,1. <=Good,2. No,75.043154
1,1,2004,24,1. Never Married,1. White,4. College Grad,2. Information,2. >=Very Good,2. No,70.476020
2,2,2003,45,2. Married,1. White,3. Some College,1. Industrial,1. <=Good,1. Yes,130.982177
3,3,2003,43,2. Married,3. Asian,4. College Grad,2. Information,2. >=Very Good,1. Yes,154.685293
4,4,2005,50,4. Divorced,1. White,2. HS Grad,2. Information,1. <=Good,1. Yes,75.043154


## Análise exploratória

### Número de registros e variáveis

In [13]:
df.shape, df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  3000 non-null   int64  
 1   year        3000 non-null   int64  
 2   age         3000 non-null   int64  
 3   maritl      3000 non-null   object 
 4   race        3000 non-null   object 
 5   education   3000 non-null   object 
 6   jobclass    3000 non-null   object 
 7   health      3000 non-null   object 
 8   health_ins  3000 non-null   object 
 9   wage        3000 non-null   float64
dtypes: float64(1), int64(3), object(6)
memory usage: 234.5+ KB


((3000, 10), None)

São 3000 registros e 10 variáveis, sendo 1 variável alvo (wage), 1 variável de index (Unnamed: 0) e 8 variáveis preditoras. A variável de index não será utilizada na modelagem, portanto, será descartada.

### Valores ausentes

In [12]:
df.isnull().sum()

Unnamed: 0    0
year          0
age           0
maritl        0
race          0
education     0
jobclass      0
health        0
health_ins    0
wage          0
dtype: int64

Não há valores ausentes no dataset.

### Análise da variável alvo

In [16]:
alvo = df['wage']

display(alvo.mean(), alvo.median())

np.float64(111.70360820174345)

104.921506533664

A média da variável alvo é maior que a mediana, o que indica que a distribuição da variável alvo é assimétrica à direita.

In [19]:
q1 = alvo.quantile(0.25)
q3 = alvo.quantile(0.75)
iqr = q3 - q1

alvo_outliers_moderados = alvo[(alvo < (q1 - 1.5 * iqr)) | (alvo > (q3 + 1.5 * iqr))]
alvo_outliers_extremos = alvo[(alvo < (q1 - 3 * iqr)) | (alvo > (q3 + 3 * iqr))]
alvo_outliers_moderados.count(), alvo_outliers_extremos.count()

(np.int64(110), np.int64(77))

Há 110 outliers moderados e 77 outliers extremos na variável alvo.

In [22]:
# Exibindo alvo_outliers_extremos em ordem decrescente
display(alvo_outliers_extremos.sort_values(ascending=False))

206     318.342430
1229    318.342430
503     314.329336
1293    311.934569
1112    309.571771
           ...    
916     267.901087
2351    267.901087
2523    267.901087
2694    267.901087
2834    267.901087
Name: wage, Length: 77, dtype: float64

Como todos os outliers são valores altos, e factívelmente representam salários altos, não serão removidos do dataset.

## Pré-processamento

### Remoção da variável de index

In [28]:
df = df.drop(columns=['Unnamed: 0'])
display(df.head())

,year,age,maritl,race,education,jobclass,health,health_ins,wage
0,2006,18,1. Never Married,1. White,1. < HS Grad,1. Industrial,1. <=Good,2. No,75.043154
1,2004,24,1. Never Married,1. White,4. College Grad,2. Information,2. >=Very Good,2. No,70.476020
2,2003,45,2. Married,1. White,3. Some College,1. Industrial,1. <=Good,1. Yes,130.982177
3,2003,43,2. Married,3. Asian,4. College Grad,2. Information,2. >=Very Good,1. Yes,154.685293
4,2005,50,4. Divorced,1. White,2. HS Grad,2. Information,1. <=Good,1. Yes,75.043154


### Normalização e codificação das variáveis preditoras

In [37]:
df_preditoras = df.drop(columns=['wage'])

df_numericas = df_preditoras.select_dtypes(include=[np.number])
df_categoricas = df_preditoras.select_dtypes(include=[object])

In [38]:
scaler = MinMaxScaler()
scaler.fit(df_numericas)
df_numericas_normalizadas = scaler.transform(df_numericas)
df_numericas_normalizadas = pd.DataFrame(df_numericas_normalizadas, columns=df_numericas.columns)
display(df_numericas_normalizadas.head())

,year,age
0,0.500000,0.000000
1,0.166667,0.096774
2,0.000000,0.435484
3,0.000000,0.403226
4,0.333333,0.516129


In [36]:
df_categoricas_encoded = pd.get_dummies(df_categoricas, drop_first=True)
display(df_categoricas_encoded.head())

,maritl_2. Married,maritl_3. Widowed,maritl_4. Divorced,maritl_5. Separated,race_2. Black,race_3. Asian,race_4. Other,education_2. HS Grad,education_3. Some College,education_4. College Grad,education_5. Advanced Degree,jobclass_2. Information,health_2. >=Very Good,health_ins_2. No
0,False,False,False,False,False,False,False,False,False,False,False,False,False,True
1,False,False,False,False,False,False,False,False,False,True,False,True,True,True
2,True,False,False,False,False,False,False,False,True,False,False,False,False,False
3,True,False,False,False,False,True,False,False,False,True,False,True,True,False
4,False,False,True,False,False,False,False,True,False,False,False,True,False,False


In [40]:
X = pd.concat([df_numericas_normalizadas, df_categoricas_encoded], axis=1)
y = alvo
display(X.head(), y.head())

,year,age,maritl_2. Married,maritl_3. Widowed,maritl_4. Divorced,maritl_5. Separated,race_2. Black,race_3. Asian,race_4. Other,education_2. HS Grad,education_3. Some College,education_4. College Grad,education_5. Advanced Degree,jobclass_2. Information,health_2. >=Very Good,health_ins_2. No
0,0.500000,0.000000,False,False,False,False,False,False,False,False,False,False,False,False,False,True
1,0.166667,0.096774,False,False,False,False,False,False,False,False,False,True,False,True,True,True
2,0.000000,0.435484,True,False,False,False,False,False,False,False,True,False,False,False,False,False
3,0.000000,0.403226,True,False,False,False,False,True,False,False,False,True,False,True,True,False
4,0.333333,0.516129,False,False,True,False,False,False,False,True,False,False,False,True,False,False


0     75.043154
1     70.476020
2    130.982177
3    154.685293
4     75.043154
Name: wage, dtype: float64